# Tic-Tac-Toe — Q-Learning Agent

A reinforcement-learning agent trained with Q-learning that **never loses** against a smart opponent.

**Key fixes over the original:**
- Plain board string used as state key (no broken symmetry transform)
- Backward eligibility-trace: when the agent loses, **all** moves in that game are penalized, not just the last
- Blocking reward (+3) and fork reward (+2) for richer shaping
- 500,000 training episodes against a priority-driven smart opponent

**Run order:**
1. **Cell 1** — imports, utilities, agent definition, and training (run once; saves `q_table_fixed.pkl`)
2. **Cell 2** — Human vs Human
3. **Cell 3** — Human vs Computer

In [2]:
import random, pickle, os

# ─── Board utilities ────────────────────────────────────────────────────────

def make_board():
    """Fresh 3x3 board; integers 1-9 mark empty cells."""
    return [[(r * 3 + c + 1) for c in range(3)] for r in range(3)]

def draw_board(board):
    print("+-------+-------+-------+")
    for row in range(3):
        print("|       |       |       |")
        print("|  ", board[row][0], "  |  ", board[row][1], "  |  ", board[row][2], "  |")
        print("|       |       |       |")
        print("+-------+-------+-------+")

def get_free_cells(board):
    return [(r, c) for r in range(3) for c in range(3) if isinstance(board[r][c], int)]

def check_win(board, sign):
    b = board
    for i in range(3):
        if all(b[i][j] == sign for j in range(3)): return True
        if all(b[j][i] == sign for j in range(3)): return True
    if b[0][0] == b[1][1] == b[2][2] == sign: return True
    if b[0][2] == b[1][1] == b[2][0] == sign: return True
    return False

def is_full(board):
    return all(not isinstance(board[r][c], int) for r in range(3) for c in range(3))

def board_to_str(board):
    """String key for Q-table. No symmetry collapsing — that was the old bug."""
    return ''.join(str(board[r][c]) for r in range(3) for c in range(3))

def find_winning_move(board, sign):
    for (r, c) in get_free_cells(board):
        board[r][c] = sign
        won = check_win(board, sign)
        board[r][c] = (r * 3 + c + 1)
        if won: return (r, c)
    return None

def count_winning_moves(board, sign):
    count = 0
    for (r, c) in get_free_cells(board):
        board[r][c] = sign
        if check_win(board, sign): count += 1
        board[r][c] = (r * 3 + c + 1)
    return count

# ─── Smart training opponent ─────────────────────────────────────────────────
# Priority: win > block > fork > block-fork > center > corners > edges

def smart_opponent_move(board, opp_sign='O', ai_sign='X'):
    free = get_free_cells(board)
    if not free: return None

    move = find_winning_move(board, opp_sign)
    if move: return move
    move = find_winning_move(board, ai_sign)
    if move: return move

    for (r, c) in free:
        board[r][c] = opp_sign
        if count_winning_moves(board, opp_sign) >= 2:
            board[r][c] = (r * 3 + c + 1); return (r, c)
        board[r][c] = (r * 3 + c + 1)

    for (r, c) in free:
        board[r][c] = ai_sign
        if count_winning_moves(board, ai_sign) >= 2:
            board[r][c] = (r * 3 + c + 1); return (r, c)
        board[r][c] = (r * 3 + c + 1)

    for pos in [(1,1),(0,0),(0,2),(2,0),(2,2),(0,1),(1,0),(1,2),(2,1)]:
        if pos in free: return pos
    return random.choice(free)

# ─── Q-Learning Agent ────────────────────────────────────────────────────────

class QLearningAgent:
    """
    Q(s,a) <- Q(s,a) + alpha * (r + gamma * max_Q(s') - Q(s,a))
    alpha=0.3  gamma=0.95  epsilon: 1.0 -> 0.01 over training
    """
    def __init__(self, alpha=0.3, gamma=0.95,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.999985):
        self.q_table       = {}
        self.alpha         = alpha
        self.gamma         = gamma
        self.epsilon       = epsilon_start
        self.epsilon_end   = epsilon_end
        self.epsilon_decay = epsilon_decay

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def update_q(self, state, action, reward, next_state, next_free):
        max_next = max([self.get_q(next_state, a) for a in next_free], default=0.0)
        old_q = self.get_q(state, action)
        self.q_table[(state, action)] = old_q + self.alpha * (reward + self.gamma * max_next - old_q)

    def penalize_game_history(self, history, base_penalty=-10):
        """Eligibility trace: penalize all moves, earlier ones less (gamma-decayed)."""
        penalty = base_penalty
        for (state, action) in reversed(history):
            old_q = self.get_q(state, action)
            self.q_table[(state, action)] = old_q + self.alpha * (penalty - old_q)
            penalty *= self.gamma

    def choose_action(self, board, training=True):
        """Epsilon-greedy: random during training, best-Q during play."""
        state = board_to_str(board)
        free  = get_free_cells(board)
        if not free: return None
        if training and random.random() < self.epsilon:
            return random.choice(free)
        q_vals = [self.get_q(state, a) for a in free]
        max_q  = max(q_vals)
        return random.choice([a for a, q in zip(free, q_vals) if q == max_q])

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

    def train(self, episodes=500_000, smart_opp_ratio=0.85):
        """
        Rewards: win +10 | block +3 | fork +2 | draw +0.5 | lose -10 (all moves)
        AI always opens in center. 85% of games use the smart opponent.
        """
        print(f"Training for {episodes:,} episodes...")
        wins = draws = losses = 0

        for ep in range(1, episodes + 1):
            board = make_board()
            board[1][1] = 'X'
            game_history = [(board_to_str(board), (1, 1))]
            use_smart = random.random() < smart_opp_ratio
            game_over = False

            while not game_over:
                # Opponent (O)
                free = get_free_cells(board)
                if not free: draws += 1; break
                opp_move = smart_opponent_move(board) if use_smart else random.choice(free)
                if opp_move is None: draws += 1; break
                board[opp_move[0]][opp_move[1]] = 'O'
                if check_win(board, 'O'):
                    self.penalize_game_history(game_history)
                    losses += 1; game_over = True; break
                if is_full(board):
                    if game_history:
                        s, a = game_history[-1]
                        self.update_q(s, a, 0.5, board_to_str(board), [])
                    draws += 1; game_over = True; break

                # Agent (X)
                state  = board_to_str(board)
                action = self.choose_action(board, training=True)
                if action is None: draws += 1; break

                reward = 0.0
                board[action[0]][action[1]] = 'O'
                if check_win(board, 'O'): reward += 3.0
                board[action[0]][action[1]] = (action[0] * 3 + action[1] + 1)

                board[action[0]][action[1]] = 'X'
                next_state = board_to_str(board)
                next_free  = get_free_cells(board)

                if check_win(board, 'X'):
                    self.update_q(state, action, 10, next_state, [])
                    wins += 1; game_over = True
                elif is_full(board):
                    self.update_q(state, action, 0.5, next_state, [])
                    draws += 1; game_over = True
                else:
                    if count_winning_moves(board, 'X') >= 2: reward += 2.0
                    self.update_q(state, action, reward, next_state, next_free)

                game_history.append((state, action))
            self.decay_epsilon()

            if ep % 100_000 == 0:
                total = wins + draws + losses
                print(f"  ep {ep:>7,} | entries {len(self.q_table):>6,} "
                      f"| W {wins/total*100:.1f}%  D {draws/total*100:.1f}%  L {losses/total*100:.1f}%"
                      f"  e={self.epsilon:.4f}")
                wins = draws = losses = 0

        print(f"\nTraining complete -- {len(self.q_table):,} (state, action) pairs learned.")

    def benchmark(self, games=5000, smart=True):
        """Report win/draw/loss rates over `games` episodes."""
        wins = draws = losses = 0
        for _ in range(games):
            board = make_board(); board[1][1] = 'X'
            game_over = False
            while not game_over:
                free = get_free_cells(board)
                if not free: draws += 1; break
                opp_move = smart_opponent_move(board) if smart else random.choice(free)
                if opp_move is None: draws += 1; break
                board[opp_move[0]][opp_move[1]] = 'O'
                if check_win(board, 'O'):  losses += 1; game_over = True; break
                if is_full(board):         draws  += 1; game_over = True; break
                ai_move = self.choose_action(board, training=False)
                if ai_move is None: draws += 1; break
                board[ai_move[0]][ai_move[1]] = 'X'
                if check_win(board, 'X'):  wins  += 1; game_over = True
                elif is_full(board):       draws += 1; game_over = True
        total = wins + draws + losses
        label = 'smart' if smart else 'random'
        print(f"Benchmark vs {label} ({games:,} games): "
              f"W {wins/total*100:.1f}%  D {draws/total*100:.1f}%  L {losses/total*100:.1f}%")

    def save(self, path='q_table_fixed.pkl'):
        with open(path, 'wb') as f:
            pickle.dump({'q_table': self.q_table, 'epsilon': self.epsilon}, f)
        print(f"Saved to '{path}' ({len(self.q_table):,} entries)")

    def load(self, path='q_table_fixed.pkl'):
        if os.path.exists(path):
            with open(path, 'rb') as f:
                data = pickle.load(f)
            self.q_table = data['q_table']
            self.epsilon = data.get('epsilon', self.epsilon_end)
            print(f"Loaded '{path}' ({len(self.q_table):,} entries)")
            return True
        print("No saved Q-table found -- training from scratch.")
        return False


# ─── Train / load ────────────────────────────────────────────────────────────

Q_TABLE_FILE = 'q_table_fixed.pkl'
agent = QLearningAgent()

if not agent.load(Q_TABLE_FILE):
    agent.train(episodes=500_000, smart_opp_ratio=0.85)
    agent.save(Q_TABLE_FILE)
else:
    print("Using saved Q-table. Delete q_table_fixed.pkl to retrain.")

agent.benchmark(5000, smart=False)
agent.benchmark(5000, smart=True)


No saved Q-table found -- training from scratch.
Training for 500,000 episodes...
  ep 100,000 | entries  2,284 | W 12.6%  D 59.6%  L 27.7%  e=0.2231
  ep 200,000 | entries  2,290 | W 14.4%  D 80.9%  L 4.7%  e=0.0498
  ep 300,000 | entries  2,291 | W 14.8%  D 84.3%  L 1.0%  e=0.0111
  ep 400,000 | entries  2,291 | W 14.5%  D 85.2%  L 0.4%  e=0.0100
  ep 500,000 | entries  2,291 | W 14.6%  D 85.0%  L 0.4%  e=0.0100

Training complete -- 2,291 (state, action) pairs learned.
Saved to 'q_table_fixed.pkl' (2,291 entries)
Benchmark vs random (5,000 games): W 96.5%  D 3.5%  L 0.0%
Benchmark vs smart (5,000 games): W 0.0%  D 100.0%  L 0.0%


## Human vs Human

Two players alternate entering a number (1–9) matching the cell position shown on the board.
Player 1 is **X**, Player 2 is **O**.

In [3]:
def play_human_vs_human():
    board   = make_board()
    current = 'X'
    print("\n" + "="*40)
    print("   TIC-TAC-TOE  |  Player 1: X   Player 2: O")
    print("="*40)
    draw_board(board)

    while True:
        print(f"Player {'1' if current == 'X' else '2'} ({current})'s turn")
        while True:
            try:
                move = int(input("Enter move (1-9): ")) - 1
                r, c = move // 3, move % 3
                if not (0 <= move <= 8):
                    print("Enter a number between 1 and 9."); continue
                if not isinstance(board[r][c], int):
                    print("Cell already taken!"); continue
                break
            except ValueError:
                print("Invalid input.")

        board[r][c] = current
        draw_board(board)

        if check_win(board, current):
            print(f"Player {'1' if current == 'X' else '2'} ({current}) wins!")
            break
        if is_full(board):
            print("It's a draw!"); break

        current = 'O' if current == 'X' else 'X'

play_human_vs_human()



   TIC-TAC-TOE  |  Player 1: X   Player 2: O
+-------+-------+-------+
|       |       |       |
|   1   |   2   |   3   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   4   |   5   |   6   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   7   |   8   |   9   |
|       |       |       |
+-------+-------+-------+
Player 1 (X)'s turn
+-------+-------+-------+
|       |       |       |
|   X   |   2   |   3   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   4   |   5   |   6   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   7   |   8   |   9   |
|       |       |       |
+-------+-------+-------+
Player 2 (O)'s turn
+-------+-------+-------+
|       |       |       |
|   X   |   O   |   3   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   4   |   5   |   6   |
|       |       |       |
+-------+-------+-------+
|   

## Human vs Computer

You play as **O**; the trained agent plays as **X** and always opens in the center.
Enter a number (1–9) to place your piece.

In [6]:
def play_human_vs_computer():
    board = make_board()
    board[1][1] = 'X'
    print("\n" + "="*40)
    print("   TIC-TAC-TOE  |  You: O   AI: X")
    print("="*40)
    print("AI opens in the center.")
    draw_board(board)

    while True:
        # Human (O)
        while True:
            try:
                move = int(input("Your move (1-9): ")) - 1
                r, c = move // 3, move % 3
                if not (0 <= move <= 8):
                    print("Enter a number between 1 and 9."); continue
                if not isinstance(board[r][c], int):
                    print("That cell is already taken!"); continue
                break
            except ValueError:
                print("Invalid input.")

        board[r][c] = 'O'
        draw_board(board)
        if check_win(board, 'O'):
            print("You won! Impressive."); break
        if is_full(board):
            print("It's a draw!"); break

        # Agent (X)
        print("AI is thinking...")
        ai_move = agent.choose_action(board, training=False)
        if ai_move is None:
            print("It's a draw!"); break
        board[ai_move[0]][ai_move[1]] = 'X'
        draw_board(board)
        if check_win(board, 'X'):
            print("AI wins! Better luck next time."); break
        if is_full(board):
            print("It's a draw!"); break

play_human_vs_computer()



   TIC-TAC-TOE  |  You: O   AI: X
AI opens in the center.
+-------+-------+-------+
|       |       |       |
|   1   |   2   |   3   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   4   |   X   |   6   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   7   |   8   |   9   |
|       |       |       |
+-------+-------+-------+
+-------+-------+-------+
|       |       |       |
|   O   |   2   |   3   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   4   |   X   |   6   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   7   |   8   |   9   |
|       |       |       |
+-------+-------+-------+
AI is thinking...
+-------+-------+-------+
|       |       |       |
|   O   |   2   |   3   |
|       |       |       |
+-------+-------+-------+
|       |       |       |
|   X   |   X   |   6   |
|       |       |       |
+-------+-------+-------+
|       |    